# AI Impact on Engineering Productivity — Full Analysis

Organized by measurement category. Each section tests specific hypotheses.

## Methodology
- **Data join:** Swarmia PR data → `VW__SWARMIA_AUTHORS` (email) → ANS v4 (AI adoption stage)
- **Cohorts:** Non-user (no AI usage) | Explorer (Stage 2) | Baseline (Stage 3) | Advanced (Stage 4-5)
- **Time window:** 3 months for PRs, aligned with ANS v4's 90-day event window
- **ANS v4** classifies by concurrency patterns (15-second sliding windows), not just volume
- **Caveat:** ANS is a rolling snapshot. Correlation ≠ causation. We include a self-selection control.

In [ ]:
import snowflake.connector
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conn = snowflake.connector.connect(
    account='wt74883-sb_prod',
    user='jeroen.vaelen@happening.xyz',
    authenticator='externalbrowser'
)

def run_query(sql):
    cur = conn.cursor()
    cur.execute(sql)
    df = cur.fetch_pandas_all()
    df.columns = df.columns.str.lower()
    return df

def gini_coefficient(values):
    s = np.sort(np.array(values, dtype=float))
    n = len(s)
    if n == 0 or s.sum() == 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * s) - (n + 1) * np.sum(s)) / (n * np.sum(s)))

AREAS = ['Core Experience', 'Data', 'Gaming', 'Platform', 'Player', 'Social', 'Sports']
COHORT_ORDER = ['Non-user', 'Explorer', 'Baseline', 'Advanced']
COHORT_COLORS = {'Non-user': '#636EFA', 'Explorer': '#EF553B', 'Baseline': '#00CC96', 'Advanced': '#AB63FA'}

COHORT_CASE = """
  CASE
    WHEN ans.stage IS NULL THEN 'Non-user'
    WHEN ans.stage = 2 THEN 'Explorer'
    WHEN ans.stage = 3 THEN 'Baseline'
    WHEN ans.stage >= 4 THEN 'Advanced'
  END
""".strip()

COHORT_ORDER_SQL = "MIN(COALESCE(ans.stage, 0))"

COHORT_SORT = """CASE cohort
  WHEN 'Non-user' THEN 0 WHEN 'Explorer' THEN 1
  WHEN 'Baseline' THEN 2 WHEN 'Advanced' THEN 3
END"""

all_bot_author_ids = []  # populated in bot discovery cell

print('Connected!')

---
## Data Exploration & Join Validation

Before analysis, verify: how many PR authors map to ANS scores?

In [ ]:
# --- Join funnel ---
df_funnel = run_query("""
SELECT
  COUNT(DISTINCT pr.author_id) as total_authors,
  COUNT(DISTINCT CASE WHEN sa.IS_MAPPED = TRUE THEN pr.author_id END) as mapped_authors,
  COUNT(DISTINCT CASE WHEN sa.IS_MAPPED = TRUE AND ans.email IS NOT NULL THEN pr.author_id END) as ans_matched
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
LEFT JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
""")
total = int(df_funnel['total_authors'].iloc[0])
mapped = int(df_funnel['mapped_authors'].iloc[0])
matched = int(df_funnel['ans_matched'].iloc[0])
print(f"Join funnel: {total} PR authors → {mapped} mapped ({100*mapped/total:.0f}%) → {matched} ANS matched ({100*matched/total:.0f}%)")

# --- Cohort distribution ---
df_cohorts = run_query(f"""
SELECT {COHORT_CASE} as cohort, COUNT(DISTINCT pr.author_id) as authors
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
df_cohorts['pct'] = (100 * df_cohorts['authors'] / df_cohorts['authors'].sum()).round(1)
df_cohorts['label'] = df_cohorts.apply(lambda r: f"{int(r['authors'])} ({r['pct']:.0f}%)", axis=1)
fig = px.bar(df_cohorts, x='cohort', y='authors', text='label', title='Cohort Distribution',
             category_orders={'cohort': COHORT_ORDER}, color='cohort', color_discrete_map=COHORT_COLORS)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, yaxis_title='Authors')
fig.show()

# --- Area coverage ---
areas_str = "','".join(AREAS)
df_areas = run_query(f"""
SELECT f.value::string as area, COUNT(DISTINCT pr.author_id) as authors,
  COUNT(DISTINCT CASE WHEN ans.email IS NOT NULL THEN pr.author_id END) as ans_matched,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN ans.email IS NOT NULL THEN pr.author_id END)
    / NULLIF(COUNT(DISTINCT pr.author_id), 0), 1) as coverage_pct
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL,
LATERAL FLATTEN(input => pr.owner_team_names) f
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
  AND f.value::string IN ('{areas_str}')
GROUP BY 1 ORDER BY 1
""")
fig2 = px.bar(df_areas, x='area', y='coverage_pct', text='coverage_pct', title='ANS Coverage by Area (%)')
fig2.update_traces(textposition='outside', cliponaxis=False, texttemplate='%{text:.0f}%')
fig2.update_layout(yaxis_title='Coverage %', yaxis_range=[0, 110])
fig2.show()
low = df_areas[df_areas['coverage_pct'] < 50]
if len(low) > 0:
    print(f"⚠ Low coverage (<50%): {', '.join(low['area'].tolist())}")
for _, r in df_areas.iterrows():
    print(f"  {r['area']}: {int(r['ans_matched'])}/{int(r['authors'])} authors ({r['coverage_pct']:.0f}%)")

---
## Output — Are AI Adopters Producing More?

Metrics: PR throughput (PRs/month per author), lines of code shipped

In [ ]:
df_tp = run_query(f"""
SELECT {COHORT_CASE} as cohort,
  COUNT(DISTINCT pr.author_id) as authors,
  COUNT(*) as prs,
  ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0) / 3.0, 1) as prs_per_month
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
fig = px.bar(df_tp, x='cohort', y='prs_per_month', text='prs_per_month',
             title='PRs/Month per Author by Cohort', color='cohort',
             category_orders={'cohort': COHORT_ORDER}, color_discrete_map=COHORT_COLORS)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, yaxis_title='PRs/month per author')
fig.show()
for _, r in df_tp.iterrows():
    print(f"{r['cohort']}: {r['prs_per_month']} PRs/month ({int(r['authors'])} authors, {int(r['prs'])} PRs)")

In [ ]:
df_lines = run_query(f"""
SELECT
  {COHORT_CASE} as cohort,
  COUNT(DISTINCT pr.author_id) as authors,
  SUM(pr.additions + pr.deletions) as total_lines,
  ROUND(SUM(pr.additions + pr.deletions)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0), 0) as lines_per_author,
  ROUND(MEDIAN(pr.additions + pr.deletions), 0) as median_pr_lines
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
  AND pr.additions IS NOT NULL
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
fig = px.bar(df_lines, x='cohort', y='lines_per_author', text='lines_per_author',
             title='Lines Shipped per Author (3-month total)', color='cohort',
             category_orders={'cohort': COHORT_ORDER}, color_discrete_map=COHORT_COLORS)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, yaxis_title='Lines per author')
fig.show()
for _, r in df_lines.iterrows():
    print(f"{r['cohort']}: {int(r['lines_per_author'])} lines/author, median PR size {int(r['median_pr_lines'])} lines")

In [ ]:
df_sel = run_query(f"""
WITH current_cohorts AS (
  SELECT sa.ID as author_id, {COHORT_CASE} as cohort
  FROM SANDBOX.VW__SWARMIA_AUTHORS sa
  LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
  WHERE sa.IS_MAPPED = TRUE
),
active_both AS (
  SELECT author_id
  FROM RAW_MISC.SWARMIA_PULL_REQUESTS
  WHERE pr_status = 'MERGED' AND is_excluded = FALSE
  GROUP BY 1
  HAVING MIN(github_created_at) < '2024-04-01'
    AND MAX(github_created_at) >= DATEADD('month', -3, CURRENT_DATE)
),
historical AS (
  SELECT cc.cohort,
    COUNT(DISTINCT pr.author_id) as authors,
    ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0), 1) as prs_per_author,
    ROUND(MEDIAN(pr.cycle_time_seconds) / 3600.0, 1) as median_cycle_h,
    ROUND(MEDIAN(pr.additions + pr.deletions), 0) as median_lines
  FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
  JOIN current_cohorts cc ON pr.author_id = cc.author_id
  JOIN active_both ab ON pr.author_id = ab.author_id
  WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
    AND pr.github_created_at >= '2024-01-01' AND pr.github_created_at < '2024-04-01'
    AND pr.cycle_time_seconds IS NOT NULL
  GROUP BY 1
),
recent AS (
  SELECT cc.cohort,
    COUNT(DISTINCT pr.author_id) as authors,
    ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0), 1) as prs_per_author,
    ROUND(MEDIAN(pr.cycle_time_seconds) / 3600.0, 1) as median_cycle_h,
    ROUND(MEDIAN(pr.additions + pr.deletions), 0) as median_lines
  FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
  JOIN current_cohorts cc ON pr.author_id = cc.author_id
  JOIN active_both ab ON pr.author_id = ab.author_id
  WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
    AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
    AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
    AND pr.cycle_time_seconds IS NOT NULL
  GROUP BY 1
)
SELECT
  COALESCE(h.cohort, r.cohort) as cohort,
  h.authors as hist_authors, r.authors as recent_authors,
  h.prs_per_author as hist_throughput, r.prs_per_author as recent_throughput,
  h.median_cycle_h as hist_cycle, r.median_cycle_h as recent_cycle,
  h.median_lines as hist_size, r.median_lines as recent_size
FROM historical h
FULL JOIN recent r ON h.cohort = r.cohort
ORDER BY CASE COALESCE(h.cohort, r.cohort)
  WHEN 'Non-user' THEN 0 WHEN 'Explorer' THEN 1
  WHEN 'Baseline' THEN 2 WHEN 'Advanced' THEN 3
END
""")

# Grouped bar: historical vs recent throughput
plot_df = pd.DataFrame([
    {'cohort': r['cohort'], 'period': 'Q1 2024', 'prs_per_author': r['hist_throughput']}
    for _, r in df_sel.iterrows()
] + [
    {'cohort': r['cohort'], 'period': 'Recent 3mo', 'prs_per_author': r['recent_throughput']}
    for _, r in df_sel.iterrows()
])
fig = px.bar(plot_df, x='cohort', y='prs_per_author', color='period', barmode='group',
             text='prs_per_author', title='Self-Selection Control: Q1 2024 vs Recent Throughput',
             category_orders={'cohort': COHORT_ORDER})
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(yaxis_title='PRs per author (period total)')
fig.show()

# Interpretation
for _, r in df_sel.iterrows():
    hist_t = r['hist_throughput'] or 0
    recent_t = r['recent_throughput'] or 0
    delta = recent_t - hist_t
    print(f"{r['cohort']}: Q1-2024={hist_t} → Recent={recent_t} (Δ {delta:+.1f})")
adv = df_sel[df_sel['cohort'] == 'Advanced']
non = df_sel[df_sel['cohort'] == 'Non-user']
if len(adv) > 0 and len(non) > 0:
    h_gap = (adv['hist_throughput'].iloc[0] or 0) - (non['hist_throughput'].iloc[0] or 0)
    print(f"\nAdvanced vs Non-user gap in Q1 2024: {h_gap:+.1f} PRs/author")
    if h_gap > 2:
        print("Self-selection detected: Advanced users were already more productive before AI adoption.")
    else:
        print("Minimal pre-existing gap — AI effect is more credible.")

---
## Efficiency — Are AI Adopters Faster?

Metrics: cycle time (total + progress/review/merge phases), size-controlled cycle time

In [ ]:
df_ct = run_query(f"""
SELECT {COHORT_CASE} as cohort,
  ROUND(AVG(pr.cycle_time_seconds) / 3600.0, 1) as avg_cycle_h,
  ROUND(AVG(pr.progress_time_seconds) / 3600.0, 1) as avg_progress_h,
  ROUND(AVG(pr.review_time_seconds) / 3600.0, 1) as avg_review_h,
  ROUND(AVG(pr.merge_time_seconds) / 3600.0, 1) as avg_merge_h
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.cycle_time_seconds IS NOT NULL
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
fig = go.Figure()
for phase, color in [('avg_progress_h', '#636EFA'), ('avg_review_h', '#EF553B'), ('avg_merge_h', '#00CC96')]:
    label = phase.replace('avg_', '').replace('_h', '').title()
    fig.add_trace(go.Bar(x=df_ct['cohort'], y=df_ct[phase], name=label,
                         marker_color=color, text=df_ct[phase],
                         textposition='inside', cliponaxis=False))
fig.update_layout(barmode='stack', title='Cycle Time Phases by Cohort (hours)',
                  xaxis={'categoryorder': 'array', 'categoryarray': COHORT_ORDER}, yaxis_title='Hours')
fig.show()
for _, r in df_ct.iterrows():
    total = r['avg_cycle_h']
    review_pct = 100 * r['avg_review_h'] / total if total else 0
    print(f"{r['cohort']}: {total}h total (review = {r['avg_review_h']}h, {review_pct:.0f}% of total)")

In [ ]:
df_sc = run_query(f"""
SELECT
  CASE
    WHEN pr.additions + pr.deletions <= 50 THEN 'XS (<=50)'
    WHEN pr.additions + pr.deletions <= 200 THEN 'S-M (51-200)'
    WHEN pr.additions + pr.deletions <= 400 THEN 'L (201-400)'
    ELSE 'XL (>400)'
  END as size_bucket,
  {COHORT_CASE} as cohort,
  COUNT(*) as prs,
  ROUND(AVG(pr.cycle_time_seconds) / 3600.0, 1) as avg_cycle_hours,
  ROUND(MEDIAN(pr.cycle_time_seconds) / 3600.0, 1) as median_cycle_hours
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.cycle_time_seconds IS NOT NULL AND pr.additions IS NOT NULL
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1, 2
HAVING COUNT(*) >= 10
ORDER BY CASE
    WHEN size_bucket = 'XS (<=50)' THEN 0
    WHEN size_bucket = 'S-M (51-200)' THEN 1
    WHEN size_bucket = 'L (201-400)' THEN 2
    ELSE 3
  END, {COHORT_ORDER_SQL}
""")
SIZE_ORDER = ['XS (<=50)', 'S-M (51-200)', 'L (201-400)', 'XL (>400)']
fig = px.bar(df_sc, x='size_bucket', y='median_cycle_hours', color='cohort', barmode='group',
             text='median_cycle_hours', title='Median Cycle Time by PR Size & Cohort',
             category_orders={'size_bucket': SIZE_ORDER, 'cohort': COHORT_ORDER},
             color_discrete_map=COHORT_COLORS)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(yaxis_title='Median cycle time (hours)')
fig.show()
for bucket in SIZE_ORDER:
    subset = df_sc[df_sc['size_bucket'] == bucket]
    if len(subset) > 0:
        vals = ', '.join(f"{r['cohort']}={r['median_cycle_hours']}h" for _, r in subset.iterrows())
        print(f"{bucket}: {vals}")

In [ ]:
areas_str = "','".join(AREAS)
df_area = run_query(f"""
SELECT f.value::string as area, {COHORT_CASE} as cohort,
  COUNT(DISTINCT pr.author_id) as authors,
  ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0) / 3.0, 1) as prs_per_month,
  ROUND(AVG(pr.cycle_time_seconds) / 3600.0, 1) as avg_cycle_h
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL,
LATERAL FLATTEN(input => pr.owner_team_names) f
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.cycle_time_seconds IS NOT NULL
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
  AND f.value::string IN ('{areas_str}')
GROUP BY 1, 2
HAVING COUNT(DISTINCT pr.author_id) >= 3
ORDER BY 1, {COHORT_ORDER_SQL}
""")
# Throughput heatmap
piv_tp = df_area.pivot_table(index='area', columns='cohort', values='prs_per_month')
piv_tp = piv_tp.reindex(columns=COHORT_ORDER)
fig1 = px.imshow(piv_tp.astype(float), text_auto='.1f', title='PRs/Month per Author: Area × Cohort',
                 color_continuous_scale='Blues', aspect='auto')
fig1.show()
# Cycle time heatmap
piv_ct = df_area.pivot_table(index='area', columns='cohort', values='avg_cycle_h')
piv_ct = piv_ct.reindex(columns=COHORT_ORDER)
fig2 = px.imshow(piv_ct.astype(float), text_auto='.1f', title='Avg Cycle Time (h): Area × Cohort',
                 color_continuous_scale='Reds', aspect='auto')
fig2.show()

---
## Quality — Is Quality Holding Up?

Metrics: revert rate, PR size distribution

In [ ]:
# --- Revert rate ---
df_rev = run_query(f"""
SELECT
  {COHORT_CASE} as cohort,
  COUNT(*) as total_prs,
  COUNT(CASE WHEN pr.is_revert = TRUE THEN 1 END) as reverts,
  ROUND(100.0 * COUNT(CASE WHEN pr.is_revert = TRUE THEN 1 END) / NULLIF(COUNT(*), 0), 2) as revert_rate_pct
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
fig1 = px.bar(df_rev, x='cohort', y='revert_rate_pct', text='revert_rate_pct',
              title='Revert Rate by Cohort (%)', color='cohort',
              category_orders={'cohort': COHORT_ORDER}, color_discrete_map=COHORT_COLORS)
fig1.update_traces(textposition='outside', cliponaxis=False, texttemplate='%{text:.2f}%')
fig1.update_layout(showlegend=False, yaxis_title='Revert rate %')
fig1.show()

# --- PR size distribution ---
df_size = run_query(f"""
SELECT
  {COHORT_CASE} as cohort,
  COUNT(*) as total_prs,
  ROUND(100.0 * COUNT(CASE WHEN pr.additions + pr.deletions <= 50 THEN 1 END) / COUNT(*), 1) as pct_xs,
  ROUND(100.0 * COUNT(CASE WHEN pr.additions + pr.deletions BETWEEN 51 AND 200 THEN 1 END) / COUNT(*), 1) as pct_sm,
  ROUND(100.0 * COUNT(CASE WHEN pr.additions + pr.deletions BETWEEN 201 AND 400 THEN 1 END) / COUNT(*), 1) as pct_l,
  ROUND(100.0 * COUNT(CASE WHEN pr.additions + pr.deletions > 400 THEN 1 END) / COUNT(*), 1) as pct_xl
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.additions IS NOT NULL
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY {COHORT_ORDER_SQL}
""")
size_plot = []
for _, r in df_size.iterrows():
    for bucket, pct in [('XS', r['pct_xs']), ('S-M', r['pct_sm']), ('L', r['pct_l']), ('XL', r['pct_xl'])]:
        size_plot.append({'cohort': r['cohort'], 'size': bucket, 'pct': pct})
sp_df = pd.DataFrame(size_plot)
fig2 = px.bar(sp_df, x='cohort', y='pct', color='size', barmode='stack', text='pct',
              title='PR Size Distribution by Cohort',
              category_orders={'cohort': COHORT_ORDER, 'size': ['XS', 'S-M', 'L', 'XL']})
fig2.update_traces(texttemplate='%{text:.0f}%', textposition='inside', cliponaxis=False)
fig2.update_layout(yaxis_title='% of PRs')
fig2.show()

for _, r in df_rev.iterrows():
    print(f"{r['cohort']}: revert rate {r['revert_rate_pct']}% ({int(r['reverts'])}/{int(r['total_prs'])})")
for _, r in df_size.iterrows():
    print(f"{r['cohort']}: XS={r['pct_xs']}% S-M={r['pct_sm']}% L={r['pct_l']}% XL={r['pct_xl']}%")

---
## Bot Review Impact

Do AI-powered PR review bots (Cursor Bug Bot, internal PR Review Bot) improve outcomes?

In [ ]:
# --- Bot discovery ---
df_bots = run_query("""
SELECT DISTINCT c.author_id, sa.name as author_name
FROM RAW_MISC.SWARMIA_PULL_REQUEST_COMMENTS c
LEFT JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON c.author_id = sa.ID
WHERE c.author_is_bot = TRUE
ORDER BY 2
""")
df_bot_reviewers = run_query("""
SELECT DISTINCT r.author_id, sa.name as author_name
FROM RAW_MISC.SWARMIA_PULL_REQUEST_REVIEWS r
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON r.author_id = sa.ID
WHERE sa.name ILIKE '%bot%' OR sa.name ILIKE '%cursor%'
ORDER BY 2
""")
all_bots = pd.concat([df_bots[['author_id', 'author_name']], df_bot_reviewers[['author_id', 'author_name']]]).drop_duplicates('author_id')
all_bot_author_ids = all_bots['author_id'].tolist()
print(f"Found {len(all_bots)} bot author(s):")
for _, b in all_bots.iterrows():
    print(f"  {b['author_name']} ({b['author_id']})")

if len(all_bot_author_ids) > 0:
    bot_ids_str = "','".join(str(x) for x in all_bot_author_ids)
    # --- Adoption trend ---
    df_adopt = run_query(f"""
    WITH bot_prs AS (
      SELECT DISTINCT pull_request_id FROM RAW_MISC.SWARMIA_PULL_REQUEST_COMMENTS WHERE author_is_bot = TRUE
      UNION
      SELECT DISTINCT pull_request_id FROM RAW_MISC.SWARMIA_PULL_REQUEST_REVIEWS WHERE author_id IN ('{bot_ids_str}')
    )
    SELECT DATE_TRUNC('month', pr.github_created_at)::DATE as month,
      COUNT(*) as total_prs,
      COUNT(CASE WHEN bp.pull_request_id IS NOT NULL THEN 1 END) as bot_reviewed,
      ROUND(100.0 * COUNT(CASE WHEN bp.pull_request_id IS NOT NULL THEN 1 END) / COUNT(*), 1) as pct_bot
    FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
    LEFT JOIN bot_prs bp ON pr.id = bp.pull_request_id
    WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
      AND pr.github_created_at >= DATEADD('month', -12, CURRENT_DATE)
      AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
    GROUP BY 1 ORDER BY 1
    """)
    fig = px.line(df_adopt, x='month', y='pct_bot', title='Bot Review Adoption (% of PRs)', markers=True)
    fig.update_layout(yaxis_title='% PRs with bot review')
    fig.show()

    # --- Bot impact on review + cycle time ---
    df_impact = run_query(f"""
    WITH bot_prs AS (
      SELECT DISTINCT pull_request_id FROM RAW_MISC.SWARMIA_PULL_REQUEST_COMMENTS WHERE author_is_bot = TRUE
      UNION
      SELECT DISTINCT pull_request_id FROM RAW_MISC.SWARMIA_PULL_REQUEST_REVIEWS WHERE author_id IN ('{bot_ids_str}')
    )
    SELECT
      CASE WHEN bp.pull_request_id IS NOT NULL THEN 'Bot-reviewed' ELSE 'No bot' END as group_label,
      COUNT(*) as prs,
      ROUND(AVG(DATEDIFF('second', pr.first_review_request_at, pr.first_reviewed_at)) / 3600.0, 1) as avg_first_review_h,
      ROUND(AVG(pr.review_time_seconds) / 3600.0, 1) as avg_review_h,
      ROUND(AVG(pr.cycle_time_seconds) / 3600.0, 1) as avg_cycle_h
    FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
    LEFT JOIN bot_prs bp ON pr.id = bp.pull_request_id
    WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
      AND pr.cycle_time_seconds IS NOT NULL
      AND pr.first_review_request_at IS NOT NULL
      AND pr.first_reviewed_at IS NOT NULL
      AND pr.first_reviewed_at > pr.first_review_request_at
      AND pr.github_created_at >= DATEADD('month', -6, CURRENT_DATE)
      AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
    GROUP BY 1
    """)
    for _, r in df_impact.iterrows():
        print(f"{r['group_label']}: first_review={r['avg_first_review_h']}h, review={r['avg_review_h']}h, cycle={r['avg_cycle_h']}h ({int(r['prs'])} PRs)")
else:
    print("No bots found. Check SWARMIA_PULL_REQUEST_COMMENTS/REVIEWS manually and update all_bot_author_ids.")

---
## Workforce Dynamics — How Is AI Changing Who Contributes?

Are more people contributing meaningfully, or are top contributors just doing more?

In [ ]:
df_auth = run_query(f"""
SELECT {COHORT_CASE} as cohort, pr.author_id, COUNT(*) as pr_count
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
JOIN SANDBOX.VW__SWARMIA_AUTHORS sa ON pr.author_id = sa.ID
LEFT JOIN SANDBOX.VW__AI_NATIVE_SCORE_V4 ans ON LOWER(ans.email) = sa.EMAIL
WHERE sa.IS_MAPPED = TRUE AND pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.github_created_at >= DATEADD('month', -3, CURRENT_DATE)
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1, 2
""")

# Overall Gini
overall_gini = gini_coefficient(df_auth['pr_count'].values)
print(f"Overall Gini coefficient: {overall_gini:.3f}")

# Per-cohort Gini
for cohort in COHORT_ORDER:
    subset = df_auth[df_auth['cohort'] == cohort]['pr_count'].values
    if len(subset) > 0:
        print(f"  {cohort}: Gini={gini_coefficient(subset):.3f} ({len(subset)} authors)")

# Concentration: top 10% and top 20%
all_prs = df_auth.sort_values('pr_count', ascending=False)
n = len(all_prs)
top10_prs = all_prs.head(max(1, n // 10))['pr_count'].sum()
top20_prs = all_prs.head(max(1, n // 5))['pr_count'].sum()
total_prs = all_prs['pr_count'].sum()
print(f"\nConcentration: top 10% produce {100*top10_prs/total_prs:.0f}% of PRs, top 20% produce {100*top20_prs/total_prs:.0f}%")

# Lorenz curve
sorted_prs = np.sort(df_auth['pr_count'].values)
cumulative = np.cumsum(sorted_prs) / sorted_prs.sum()
x = np.arange(1, len(cumulative) + 1) / len(cumulative)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x * 100, y=cumulative * 100, name='Actual', mode='lines'))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], name='Perfect equality', line=dict(dash='dash')))
fig.update_layout(title=f'Lorenz Curve — PR Distribution (Gini={overall_gini:.3f})',
                  xaxis_title='Cumulative % of authors', yaxis_title='Cumulative % of PRs')
fig.show()

---
## Events & Step Changes

Look for step changes in productivity around known AI milestones.

In [ ]:
AI_EVENTS = [
    ('2024-11-01', 'Opus 4.5 Launch'),
]

df_monthly = run_query("""
SELECT
  DATE_TRUNC('month', pr.github_created_at)::DATE as month,
  COUNT(*) as prs,
  COUNT(DISTINCT pr.author_id) as authors,
  ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT pr.author_id), 0), 1) as prs_per_author,
  ROUND(AVG(pr.cycle_time_seconds) / 3600.0, 1) as avg_cycle_hours,
  ROUND(MEDIAN(pr.cycle_time_seconds) / 3600.0, 1) as median_cycle_hours,
  ROUND(MEDIAN(pr.additions + pr.deletions), 0) as median_lines
FROM RAW_MISC.SWARMIA_PULL_REQUESTS pr
WHERE pr.pr_status = 'MERGED' AND pr.is_excluded = FALSE
  AND pr.cycle_time_seconds IS NOT NULL
  AND pr.github_created_at >= '2024-01-01'
  AND DATE_TRUNC('month', pr.github_created_at) < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY 1 ORDER BY 1
""")
df_monthly['month'] = pd.to_datetime(df_monthly['month'])

fig = make_subplots(rows=3, cols=1, subplot_titles=['PRs/Author per Month', 'Median Cycle Time (h)', 'Median PR Size (lines)'],
                    shared_xaxes=True, vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=df_monthly['month'], y=df_monthly['prs_per_author'], mode='lines+markers', name='PRs/author'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_monthly['month'], y=df_monthly['median_cycle_hours'], mode='lines+markers', name='Cycle time'), row=2, col=1)
fig.add_trace(go.Scatter(x=df_monthly['month'], y=df_monthly['median_lines'], mode='lines+markers', name='PR size'), row=3, col=1)

for date_str, label in AI_EVENTS:
    for row in [1, 2, 3]:
        fig.add_vline(x=date_str, line_dash='dash', line_color='red', row=row, col=1)
        fig.add_annotation(x=date_str, y=1, yref='paper', text=label, showarrow=False,
                           font=dict(size=10, color='red'), yshift=10)
fig.update_layout(height=700, title='Monthly Trends with AI Milestones')
fig.show()

# Before/after comparison
for date_str, label in AI_EVENTS:
    event_date = pd.Timestamp(date_str)
    before = df_monthly[(df_monthly['month'] >= event_date - pd.DateOffset(months=3)) & (df_monthly['month'] < event_date)]
    after = df_monthly[(df_monthly['month'] >= event_date) & (df_monthly['month'] < event_date + pd.DateOffset(months=3))]
    if len(before) > 0 and len(after) > 0:
        tp_b, tp_a = before['prs_per_author'].mean(), after['prs_per_author'].mean()
        ct_b, ct_a = before['median_cycle_hours'].mean(), after['median_cycle_hours'].mean()
        print(f"\n{label} ({date_str}):")
        print(f"  Throughput: {tp_b:.1f} → {tp_a:.1f} ({100*(tp_a-tp_b)/tp_b:+.0f}%)")
        print(f"  Cycle time: {ct_b:.1f}h → {ct_a:.1f}h ({100*(ct_a-ct_b)/ct_b:+.0f}%)")

---
## Summary

In [ ]:
print("=" * 60)
print("AI IMPACT ANALYSIS — KEY FINDINGS")
print("=" * 60)

print(f"\n1. DATA COVERAGE")
print(f"   {matched}/{total} PR authors matched to ANS v4 ({100*matched/total:.0f}%)")

print(f"\n2. COHORT DISTRIBUTION")
for _, r in df_cohorts.iterrows():
    print(f"   {r['cohort']}: {int(r['authors'])} authors ({r['pct']:.0f}%)")

print(f"\n3. THROUGHPUT (PRs/month per author)")
for _, r in df_tp.iterrows():
    print(f"   {r['cohort']}: {r['prs_per_month']}")

print(f"\n4. CYCLE TIME (avg hours)")
for _, r in df_ct.iterrows():
    print(f"   {r['cohort']}: {r['avg_cycle_h']}h (review {r['avg_review_h']}h)")

print(f"\n5. SELF-SELECTION CONTROL")
for _, r in df_sel.iterrows():
    print(f"   {r['cohort']}: Q1-2024={r['hist_throughput']} → Recent={r['recent_throughput']}")

print(f"\n6. QUALITY")
for _, r in df_rev.iterrows():
    print(f"   {r['cohort']}: revert rate {r['revert_rate_pct']}%")

print(f"\n7. BOT IMPACT")
if len(all_bot_author_ids) > 0:
    for _, r in df_impact.iterrows():
        print(f"   {r['group_label']}: cycle={r['avg_cycle_h']}h")
else:
    print("   No bots detected")

print(f"\n8. WORKFORCE DYNAMICS")
print(f"   Gini coefficient: {overall_gini:.3f}")
print(f"   Top 10% produce {100*top10_prs/total_prs:.0f}% of PRs")

print(f"\n9. CAVEATS")
print("   - ANS v4 is a rolling snapshot; cohort assignment may shift over time")
print("   - Correlation ≠ causation; self-selection control helps but doesn't eliminate bias")
print("   - Small cohort sizes in some areas may produce unreliable estimates")
print("   - Bot detection relies on author_is_bot flag + name heuristics")